<a href="https://colab.research.google.com/github/MdRimon29/borno-lm/blob/main/src/pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the tokenizer dependency.
!pip install -q sentencepiece

In [2]:
# Install the Hugging Face datasets library.
!pip install -q datasets

In [3]:
# Mount Google Drive so checkpoints persist after the Colab session ends.
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# Import the libraries required for pretraining.
import math
import random
from pathlib import Path

import numpy as np
import sentencepiece as spm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [5]:
# Set random seeds and select the available training device.
def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

set_seed(123)

print("Device:", device)

Device: cuda


In [6]:
# Create the Borno-LM directories on Google Drive.
ROOT_DIR = Path("/content/drive/MyDrive/borno-lm")

DATA_DIR = ROOT_DIR / "data"
CHECKPOINT_DIR = ROOT_DIR / "checkpoints"
TOKENIZER_DIR = ROOT_DIR / "tokenizer"

for path in [DATA_DIR, CHECKPOINT_DIR, TOKENIZER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(ROOT_DIR)

/content/drive/MyDrive/borno-lm


In [8]:
# Load a small Bangla subset from the Hugging Face corpus.
from datasets import load_dataset

dataset = load_dataset(
    "ahmed-farhanur-rashid/bn-foundational-pretrain-corpus",
    "bangla_corpus",
    split="train",
    streaming=True,
)

Resolving data files:   0%|          | 0/55 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/55 [00:00<?, ?it/s]

In [9]:
# Collect a manageable Bangla corpus for the first pretraining experiment.
TARGET_CHARS = 5_000_000

texts = []
total_chars = 0

for sample in dataset:
    text_sample = sample["text"]

    if not text_sample:
        continue

    texts.append(text_sample)
    total_chars += len(text_sample)

    if total_chars >= TARGET_CHARS:
        break

text = "\n\n".join(texts)

print(f"Documents: {len(texts):,}")
print(f"Characters: {len(text):,}")
print(text[:1000])

Documents: 476
Characters: 5,029,401
<|lang_bn|>বাংলা ভাষা (বাঙলা, বাঙ্গলা, তথা বাঙ্গালা নামেও পরিচিত) একটি ইন্দো-আর্য ভাষা, যা দক্ষিণ এশিয়ার বাঙালি জাতির প্রধান কথ্য ও লেখ্য ভাষা। মাতৃভাষীর সংখ্যায় বাংলা ইন্দো-ইউরোপীয় ভাষা পরিবারের পঞ্চম ও মোট ব্যবহারকারীর সংখ্যা অনুসারে বাংলা বিশ্বের ষষ্ঠ বৃহত্তম ভাষা। বাংলা সার্বভৌম ভাষাভিত্তিক জাতিরাষ্ট্র বাংলাদেশের একমাত্র রাষ্ট্রভাষা তথা সরকারি ভাষা এবং ভারতের পশ্চিমবঙ্গ, ত্রিপুরা, আসামের বরাক উপত্যকার সরকারি ভাষা। বঙ্গোপসাগরে অবস্থিত আন্দামান দ্বীপপুঞ্জের প্রধান কথ্য ভাষা বাংলা। এছাড়া ভারতের ঝাড়খণ্ড, বিহার, মেঘালয়, মিজোরাম,  ওড়িশা রাজ্যগুলোতে উল্লেখযোগ্য পরিমাণে বাংলাভাষী জনগণ রয়েছে। ২০১১ সালের আদমশুমারি অনুযায়ী, ভারতের মোট জনসংখ্যার ৮.০৩ শতাংশ মানুষ বাংলা ভাষায় কথা বলে এবং হিন্দির পরেই ভারতে সর্বাধিক প্রচলিত ভাষা - বাংলা। এছাড়াও মধ্য প্রাচ্য, আমেরিকা ও ইউরোপে উল্লেখযোগ্য পরিমাণে বাংলাভাষী অভিবাসী রয়েছে। সারা বিশ্বে সব মিলিয়ে ২৭.৬ কোটির অধিক লোক দৈনন্দিন জীবনে বাংলা ব্যবহার করে। বাংলাদেশের জাতীয় সঙ্গীত এবং ভারতের জাতীয় সঙ্গীত ও স্তোত্র বাংলাতে র

In [10]:
# Split the corpus into training and validation text.
split_idx = int(len(text) * 0.85)

train_text = text[:split_idx]
val_text = text[split_idx:]

print(f"Train characters: {len(train_text):,}")
print(f"Validation characters: {len(val_text):,}")

Train characters: 4,274,990
Validation characters: 754,411


In [11]:
# Save the collected Bangla corpus and define its path.
DATA_PATH = DATA_DIR / "bangla_corpus.txt"

DATA_PATH.write_text(text, encoding="utf-8")

print(f"Dataset saved: {DATA_PATH}")
print(f"File exists: {DATA_PATH.exists()}")

Dataset saved: /content/drive/MyDrive/borno-lm/data/bangla_corpus.txt
File exists: True


In [12]:
# Train a 16K SentencePiece BPE tokenizer for Bangla.
TOKENIZER_PREFIX = TOKENIZER_DIR / "borno_bpe"
TOKENIZER_MODEL = TOKENIZER_DIR / "borno_bpe.model"

if not TOKENIZER_MODEL.exists():
    spm.SentencePieceTrainer.train(
        input=str(DATA_PATH),
        model_prefix=str(TOKENIZER_PREFIX),
        vocab_size=16_000,
        model_type="bpe",
        character_coverage=0.9995,
        normalization_rule_name="nmt_nfkc",
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
    )

print("Tokenizer:", TOKENIZER_MODEL)

Tokenizer: /content/drive/MyDrive/borno-lm/tokenizer/borno_bpe.model


In [13]:
# Define the tokenizer used by Borno-LM.
class BornoTokenizer:

    def __init__(self, model_path):
        self.sp = spm.SentencePieceProcessor(model_file=str(model_path))
        self.vocab_size = self.sp.get_piece_size()

    def encode(self, text):
        return self.sp.encode(text, out_type=int)

    def decode(self, token_ids):
        return self.sp.decode(token_ids)

    def encode_tensor(self, text, device="cpu"):
        ids = self.encode(text)

        return torch.tensor(
            ids,
            dtype=torch.long,
            device=device,
        ).unsqueeze(0)

    def decode_tensor(self, token_ids):
        return self.decode(
            token_ids.squeeze(0).tolist()
        )


tokenizer = BornoTokenizer(TOKENIZER_MODEL)

print("Vocabulary size:", tokenizer.vocab_size)

Vocabulary size: 16000


In [14]:
# Verify that Bangla text can be encoded and decoded correctly.
sample = "বাংলাদেশের রাজধানী ঢাকা।"

ids = tokenizer.encode(sample)

print("Token IDs:", ids)
print("Decoded:", tokenizer.decode(ids))

Token IDs: [112, 847, 731, 15939]
Decoded: বাংলাদেশের রাজধানী ঢাকা।


In [15]:
# Tokenize the training and validation text.
train_tokens = tokenizer.encode(train_text)
val_tokens = tokenizer.encode(val_text)

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")

Train tokens: 1,161,395
Validation tokens: 198,517


In [17]:
# Define the approximately 152M parameter Borno-LM configuration.
BORNO_CONFIG = {
    "vocab_size": tokenizer.vocab_size,
    "context_length": 512,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 18,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

In [18]:
# Create training sequences using the next-token prediction objective.
class BornoDatasetV1(Dataset):

    def __init__(self, token_ids, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return (
            self.input_ids[idx],
            self.target_ids[idx],
        )

In [19]:
# Create the training and validation datasets.
train_dataset = BornoDatasetV1(
    train_tokens,
    BORNO_CONFIG["context_length"],
    BORNO_CONFIG["context_length"],
)

val_dataset = BornoDatasetV1(
    val_tokens,
    BORNO_CONFIG["context_length"],
    BORNO_CONFIG["context_length"],
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Train samples: 2268
Validation samples: 387


In [ ]:
# Create DataLoaders for GPU training.
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    drop_last=False,
    num_workers=2,
    pin_memory=True,
)